# Type-constrained generation — LC-QuAD eval on Colab GPU

Runs `test.py` (all 1000 LC-QuAD test questions through the constrained generator) on a Colab GPU.

**One-time setup before running:**

1. Upload two files to a Google Drive folder (default expected: `MyDrive/ogd/`):
   - `qwen_lcquad.safetensors` (3.1 GB) — from your local `model/` folder
   - `class_tries.pkl` (197 MB) — from your local `dbpedia/` folder
   - (`entities.pkl` is NOT needed — generation only reads the tries)
2. Add a GitHub token as a Colab secret (repo is private): left sidebar → key icon → Secrets → name `GITHUB_TOKEN`.
   - Alternatively make the repo public and comment out the token line in the clone cell.
3. Runtime → Change runtime type → GPU.

Then Runtime → Run all. output.json is mirrored to Drive every 2 minutes while the eval runs, and the final numbers print at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # re-running this cell fixes dropped Drive connections

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ogd')  # where you uploaded the two files
REPO_DIR = Path('/content/ontologically-guided-decoding')

assert (DRIVE_DIR / 'qwen_lcquad.safetensors').exists(), 'weights not found in Drive folder'
assert (DRIVE_DIR / 'class_tries.pkl').exists(), 'class_tries.pkl not found in Drive folder'

Mounted at /content/drive


In [2]:
!git clone https://github.com/JosephMuddle/ontologically-guided-decoding.git {REPO_DIR}
!git -C /content/ontologically-guided-decoding pull


Cloning into '/content/ontologically-guided-decoding'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 116 (delta 64), reused 85 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 1.70 MiB | 12.26 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Already up to date.


In [3]:
import shutil
(REPO_DIR / 'model').mkdir(exist_ok=True)
(REPO_DIR / 'dbpedia').mkdir(exist_ok=True)
shutil.copy(DRIVE_DIR / 'qwen_lcquad.safetensors', REPO_DIR / 'model' / 'qwen_lcquad.safetensors')
shutil.copy(DRIVE_DIR / 'class_tries.pkl', REPO_DIR / 'dbpedia' / 'class_tries.pkl')
print('weights and tries copied to VM local disk')

# Resume support. test.py reads output.json from the REPO directory on this VM,
# never from Drive -- Drive is only where cell 6 mirrors it so a dropped session
# does not lose the run. So deleting the Drive copy on its own resets nothing,
# and the mirror will put it straight back. This cell is the one place the two
# are reconciled, so re-run it (not just the eval cell) whenever you change your
# mind about resuming.
RESUME = True
out = REPO_DIR / 'output.json'
drive_out = DRIVE_DIR / 'output.json'
if not RESUME:
    for f in (out, drive_out):
        if f.exists():
            f.unlink()
    print('RESUME is False: cleared both copies, starting from question 1')
elif drive_out.exists():
    import json
    shutil.copy(drive_out, out)
    n = len(json.loads(out.read_text(encoding='utf-8')))
    print(f'resuming: {n} records restored from Drive, {1000 - n} questions left')
elif out.exists():
    import json
    n = len(json.loads(out.read_text(encoding='utf-8')))
    print(f'no copy in Drive, but this VM already holds {n} records -- resuming from those.')
    print('set RESUME = False and re-run this cell to start from question 1 instead')
else:
    print('no output.json on the VM or in Drive; starting from question 1')

weights and tries copied to VM local disk
no output.json on the VM or in Drive; starting from question 1


In [4]:
# .env holds only paths here (no secrets); config and tokenizer come from the
# public base model Qwen/Qwen2.5-Coder-1.5B, only the weights are local
(REPO_DIR / '.env').write_text('MODEL_WEIGHTS=model/qwen_lcquad.safetensors\nDATA_PATH=dbpedia\n')
!pip install -q xgrammar safetensors
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 111.3 MB/s eta 0:00:00
deps installed


In [5]:
import torch, time
print('GPU:', torch.cuda.get_device_name(0))
%cd {REPO_DIR}
t0 = time.time()
from type_constrained_generation import generate_positive  # module-level: loads weights, grammars, tries
print(f'setup took {time.time() - t0:.0f}s')
print(generate_positive('What is the region of Tom Perriello ?'))  # smoke test: one constrained query

GPU: NVIDIA A100-SXM4-80GB
/content/ontologically-guided-decoding


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

loaded qwen_lcquad.safetensors into Qwen/Qwen2.5-Coder-1.5B on cuda


tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

compiled grammars: 5 beginning templates, 616 relations + type tail, 763 classes
loaded 762 class tries in 11.5s
setup took 117s
SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Tom_Perriello> <http://dbpedia.org/ontology/region> ?uri }


In [6]:
# mirror output.json to Drive every 2 min so progress survives a disconnect
!nohup bash -c 'while true; do cp -f /content/ontologically-guided-decoding/output.json /content/drive/MyDrive/ogd/output.json 2>/dev/null; sleep 120; done' > /dev/null 2>&1 &
print('mirroring output.json to Drive every 2 min')

mirroring output.json to Drive every 2 min


In [7]:
# The eval: the three guided rungs -- positive, negative and both. no_boosts,
# grammar_only and unconstrained take none of the ontological guidance, so they
# are left out here; set ONLY = '' to run all six. A rung that already has
# answers in output.json is skipped, which is what makes a timed-out run
# resumable -- and that includes a positive rung restored from an older file,
# where it is stored as 'generated' and renamed on load. If that older file was
# decoded at a different beam width, set RESUME = False in the resume cell first,
# or the positive column will not match the other two. REDO = True clears the
# selected rungs and regenerates them; leave it False on a resumed session, or
# every restart throws away the progress it just restored.
BEAMS = 3      # whole-query beam width: every rung and the baseline (1 = greedy)
ONLY = 'positive negative both'   # '' = all six rungs
REDO = False   # True to regenerate the selected rungs where answers already exist

args = f'--beams {BEAMS}'
if ONLY:
    args += f' --systems {ONLY}'
if REDO:
    args += ' --redo'
print('test.py', args)
!python /content/ontologically-guided-decoding/test.py {args}

test.py --beams 3 --systems positive negative both
loaded qwen_lcquad.safetensors into Qwen/Qwen2.5-Coder-1.5B on cuda
compiled grammars: 5 beginning templates, 616 relations + type tail, 763 classes
loaded 762 class tries in 15.8s
beam width: 3
running [positive, negative, both] on 1000 of 1000 questions
10/1000  positive 3/10 (30.0%)  mod-twins 6/10 (60.0%)  negative 3/10 (30.0%)  both 3/10 (30.0%)  12.27s per question
20/1000  positive 7/20 (35.0%)  mod-twins 11/20 (55.0%)  negative 7/20 (35.0%)  both 7/20 (35.0%)  9.27s per question
30/1000  positive 12/30 (40.0%)  mod-twins 16/30 (53.3%)  negative 11/30 (36.7%)  both 12/30 (40.0%)  8.82s per question
40/1000  positive 14/40 (35.0%)  mod-twins 19/40 (47.5%)  negative 12/40 (30.0%)  both 14/40 (35.0%)  8.68s per question
50/1000  positive 17/50 (34.0%)  mod-twins 24/50 (48.0%)  negative 13/50 (26.0%)  both 17/50 (34.0%)  8.56s per question
60/1000  positive 20/60 (33.3%)  mod-twins 28/60 (46.7%)  negative 16/60 (26.7%)  both 20/60 (

In [8]:
import json, shutil
shutil.copy(REPO_DIR / 'output.json', DRIVE_DIR / 'output.json')  # final copy
res = json.loads((REPO_DIR / 'output.json').read_text(encoding='utf-8'))

# the ablation ladder, guided rungs first: each of positive / negative / both
# minus no_boosts isolates that guidance, no_boosts - grammar_only the KB vocabulary
for name in ('positive', 'negative', 'both', 'no_boosts', 'grammar_only', 'unconstrained'):
    have = [r for r in res if f'{name}_match' in r]   # a rung may not have been run
    if not have:
        print(f'{name:14} not run')
        continue
    hits = sum(r[f'{name}_match'] for r in have)
    print(f'{name:14} {hits}/{len(have)} ({hits / len(have):.1%})')
twin = sum(r['match_modulo_twins'] for r in res)
print(f'{"mod-twins":14} {twin}/{len(res)} ({twin / len(res):.1%})  (positive rung only)')

positive       267/1000 (26.7%)
negative       265/1000 (26.5%)
both           267/1000 (26.7%)
no_boosts      not run
grammar_only   not run
unconstrained  not run
mod-twins      367/1000 (36.7%)  (positive rung only)


In [ ]:
!git -C /content/ontologically-guided-decoding log --oneline -1
!grep -c "bitmask.to(DEVICE)" /content/ontologically-guided-decoding/type_constrained_generation.py

86e1540 (HEAD -> main, origin/main, origin/HEAD) UNCONSTRAINED BASELINE
2


In [ ]:
!git -C /content/ontologically-guided-decoding pull


Already up to date.
